In [ ]:
!mkdir -p ~/.kaggle
!cp /kaggle.json ~/.kaggle

In [ ]:
!kaggle datasets download -d birajsth/cats-and-dogs-filtered

In [ ]:
import zipfile
# zip_ref = zipfile.ZipFile("/content/cats-and-dogs.zip")
zip_ref = zipfile.ZipFile("/content/cats-and-dogs-filtered.zip")
zip_ref.extractall('/content')
zip_ref.close()

In [ ]:
from keras.applications import vgg16
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Flatten,GlobalAveragePooling2D
from keras.applications.vgg16 import VGG16

base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(150,150,3)
)

base_model.summary()

model = Sequential()

model.add(base_model)
model.add(GlobalAveragePooling2D())
# model.add(Flatten())
model.add(Dense(256,activation='relu'))
model.add(Dense(1,activation='sigmoid'))
# model.summary()

base_model.trainable = False

train = keras.utils.image_dataset_from_directory(
    # directory = '/content/train', cat vs dog old dataset
    directory = '/content/cats_and_dogs_filtered/train',
    labels= 'inferred',
    label_mode = 'int',
    batch_size=32,
    image_size=(150,150)
)

test = keras.utils.image_dataset_from_directory(
    # directory = '/content/val',
    directory = '/content/cats_and_dogs_filtered/validation',
    labels= 'inferred',
    label_mode = 'int',
    batch_size=32,
    image_size=(150,150)
)

In [ ]:
# normalization
def process(image,label):
    image = tensorflow.cast(image/255. ,tensorflow.float32)
    return image,label

train = train.map(process)
test = test.map(process)

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
history = model.fit(train,epochs=10,validation_data=test)


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'],color='red',label='train')
plt.plot(history.history['val_accuracy'],color='blue',label='validation')
plt.legend()
plt.show()

In [ ]:
from keras.applications import vgg16
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.optimizers import Adam
from keras.layers import Dense,Flatten,GlobalAveragePooling2D
from keras.applications.vgg16 import VGG16

base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(150,150,3)
)

base_model.trainable = True

set_trainable = False

for layer in base_model.layers:
  if layer.name == 'block5_conv1':
    set_trainable = True
  if set_trainable:
    layer.trainable = True
  else:
    layer.trainable = False

base_model.summary()

model = Sequential()

model.add(base_model)
model.add(GlobalAveragePooling2D())
model.add(Dense(256,activation='relu'))
model.add(Dense(1,activation='sigmoid'))
# model.summary()

train = keras.utils.image_dataset_from_directory(
    # directory = '/content/train',
    directory = '/content/cats_and_dogs_filtered/train',
    labels= 'inferred',
    label_mode = 'int',
    batch_size=32,
    image_size=(150,150)
)

test = keras.utils.image_dataset_from_directory(
    # directory = '/content/val',
    directory = '/content/cats_and_dogs_filtered/validation',
    labels= 'inferred',
    label_mode = 'int',
    batch_size=32,
    image_size=(150,150)
)

# normalization
def process(image,label):
    image = tensorflow.cast(image/255. ,tensorflow.float32)
    return image,label

train = train.map(process)
test = test.map(process)

model.compile(optimizer=Adam(learning_rate=1e-5),loss='binary_crossentropy',metrics=['accuracy'])
history = model.fit(train,epochs=10,validation_data=test)

import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'],color='red',label='train')
plt.plot(history.history['val_accuracy'],color='blue',label='validation')
plt.legend()
plt.show()